### Data Processing:
- Removing and imputing missing values
- Getting Categorical data into shape for ML algos

### Dealing with Missing Data
- missing values: NaN (not a number) or NULL
- removing missing data is easy and convenient approach; however, if we remove too many samples, it's impossible to make a reliable analysis, or will lose a lot of information.

In [87]:
import pandas as pd 
from io import StringIO

In [88]:
csv_data = '''A,B,C,D 
1.0,2.0,3.0,4.0 
5.0,6.0,,8.0
9.0,,11.0,
,,,'''

In [89]:
df = pd.read_csv(StringIO(csv_data))
df

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,9.0,NaN,11.0,NaN
3,NaN,NaN,NaN,NaN


##### Finding missing values

In [90]:
# count number of missing values per columns
print(df.isnull())
print(df.isnull().sum())

       A      B      C     D 
0  False  False  False  False
1  False  False   True  False
2  False   True  False   True
3   True   True   True   True
A     1
B     2
C     2
D     2
dtype: int64


In [91]:
# convert to numpy
np_arr = df.values
print(type(np_arr))
print(np_arr)

<class 'numpy.ndarray'>
[[ 1.  2.  3.  4.]
 [ 5.  6. nan  8.]
 [ 9. nan 11. nan]
 [nan nan nan nan]]


##### Eliminating training examples or features with missing values

In [92]:
# remove the entire rows:
df.dropna()

,A,B,C,D
0,1.0,2.0,3.0,4.0


In [93]:
# remove the entire columns:
df.dropna(axis=1)

""
0
1
2
3


In [94]:
# only drop rows that have all na values
df.dropna(how="all")

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,9.0,NaN,11.0,NaN


In [95]:
# drop rows that have < 3 existing values (mean drop 2-na rows)
df.dropna(thresh=3)

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0


In [96]:
# only drop rows where na appears in a certain column
df.dropna(subset=['C'])

,A,B,C,D
0,1.0,2.0,3.0,4.0
2,9.0,NaN,11.0,NaN


##### Imputing missing values

In [97]:
from sklearn.impute import SimpleImputer
import numpy as np

In [98]:
np_arr

array([[ 1.,  2.,  3.,  4.],
       [ 5.,  6., nan,  8.],
       [ 9., nan, 11., nan],
       [nan, nan, nan, nan]])

In [99]:
# mean technique # median, most_frequent
imputer = SimpleImputer(missing_values=np.nan, strategy="median")
imputer.fit(df.values)
imputed_data = imputer.transform(df.values)
imputed_data

array([[ 1.,  2.,  3.,  4.],
       [ 5.,  6.,  7.,  8.],
       [ 9.,  4., 11.,  6.],
       [ 5.,  4.,  7.,  6.]])

In [100]:
# alternative approach for fillna
df.fillna(df.mean())

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,7.0,8.0
2,9.0,4.0,11.0,6.0
3,5.0,4.0,7.0,6.0


##### Handling Categorical data

In [101]:
df = pd.DataFrame([
    ['green', 'M', 10.1, 'class2'],
    ['red', 'XL', 8.1, 'class1'],
    ['blue', 'M', 7.1, 'class2'],
    ['gray', 'L', 9.1, 'class2'],
])

df.columns = ["color", "size", "price", "classlabel"]
df

,color,size,price,classlabel
0,green,M,10.1,class2
1,red,XL,8.1,class1
2,blue,M,7.1,class2
3,gray,L,9.1,class2


In [102]:
# mapping ordinary features:
ordinary_mapping = {"XL":3,"L":2,"M":1 }
df['size'] = df['size'].map(ordinary_mapping)
df

,color,size,price,classlabel
0,green,1,10.1,class2
1,red,3,8.1,class1
2,blue,1,7.1,class2
3,gray,2,9.1,class2


In [103]:
# reverse mapping:
reversed_ordinary_mapping = {v: k for k, v in ordinary_mapping.items()}
df['size'] = df['size'].map(reversed_ordinary_mapping)
df

,color,size,price,classlabel
0,green,M,10.1,class2
1,red,XL,8.1,class1
2,blue,M,7.1,class2
3,gray,L,9.1,class2


#### Label Encoding

##### Vanilla approach

In [104]:
# create mapping dict
class_mapping = {label: index for index, label in enumerate(np.unique(df['classlabel']))}
print(class_mapping)

{'class1': 0, 'class2': 1}


In [107]:
# mapping
df['classlabel'] = df['classlabel'].map(class_mapping)
print("mapping:\n", df)

# inverse mapping
inv_class_maping = {v: k for k, v in class_mapping.items()}
df['classlabel'] = df['classlabel'].map(inv_class_maping)
print("\ninverse mapping:\n",df)


mapping:
    color size  price  classlabel
0  green    M   10.1           1
1    red   XL    8.1           0
2   blue    M    7.1           1
3   gray    L    9.1           1

inverse mapping:
    color size  price classlabel
0  green    M   10.1     class2
1    red   XL    8.1     class1
2   blue    M    7.1     class2
3   gray    L    9.1     class2


##### Label Encoder (for ordinary label) from sklearn

In [108]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['classlabel'])
y_inverse = le.inverse_transform(y)

print(y)
print(y_inverse)

[1 0 1 1]
['class2' 'class1' 'class2' 'class2']


##### One Hot Encoder (for nominal label) from sklearn

In [117]:
from sklearn.preprocessing import OneHotEncoder

X = df[['color', 'size', 'price']].values
ohe = OneHotEncoder()
# X[:,0].reshape(-1,1)

color_ohe = ohe.fit_transform(X[:, 0].reshape(-1,1)).toarray()
color_inv_ohe = ohe.inverse_transform(color_ohe)

print(color_ohe)
print(color_inv_ohe)

[[0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]
[['green']
 ['red']
 ['blue']
 ['gray']]


##### ColumnTransformer

In [127]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
print(X)
print("pass the first column via OHE, and the other two as is")

column_transform_pipeline = ColumnTransformer([
    ('onehot', OneHotEncoder(), [0]),
    ('labelencoder', OrdinalEncoder(), [1]),
    ('nothing', 'passthrough', [2])
])

X_transformed = column_transform_pipeline.fit_transform(X)
X_transformed

[['green' 'M' 10.1]
 ['red' 'XL' 8.1]
 ['blue' 'M' 7.1]
 ['gray' 'L' 9.1]]
pass the first column via OHE, and the other two as is


array([[0.0, 0.0, 1.0, 0.0, 1.0, 10.1],
       [0.0, 0.0, 0.0, 1.0, 2.0, 8.1],
       [1.0, 0.0, 0.0, 0.0, 1.0, 7.1],
       [0.0, 1.0, 0.0, 0.0, 0.0, 9.1]], dtype=object)

In [132]:
# however, for onehotencoder, we may have the multicollineary issue -> drop one of the ohe column_transform_pipeline
new_ohe = OneHotEncoder(categories='auto', drop='first')

column_transform_pipeline = ColumnTransformer([
    ('onehot', new_ohe, [0]),
    ('labelencoder', OrdinalEncoder(), [1]),
    ('nothing', 'passthrough', [2])
])

X_transformed = column_transform_pipeline.fit_transform(X)
X_transformed

array([[0.0, 1.0, 0.0, 1.0, 10.1],
       [0.0, 0.0, 1.0, 2.0, 8.1],
       [0.0, 0.0, 0.0, 1.0, 7.1],
       [1.0, 0.0, 0.0, 0.0, 9.1]], dtype=object)

In [130]:
# for pandas: dummies, but it treats everything as nominal (only does one hot encoder)
df_transform = pd.get_dummies(df, dtype=int, drop_first=True)
df_transform

,price,color_gray,color_green,color_red,size_M,size_XL,classlabel_class2
0,10.1,0,1,0,1,0,1
1,8.1,0,0,1,0,1,0
2,7.1,0,0,0,1,0,1
3,9.1,1,0,0,0,0,1
